# Chapter 13 &mdash; Recursively Enumerable versus Recursive

**Concept 12 of the Chapter 13 decomposition:** *Recursively Enumerable versus Recursive Languages*

RE = the language of a TM; recursive = the language of a TM <i>guaranteed to halt</i>.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-RE-Versus-Recursive/Concept-RE-Versus-Recursive.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Two classes, one word of difference:

* **recursively enumerable** (RE, Turing-**recognizable**): $L = L(M)$ for some TM $M$.
  On $w\in L$ it halts and accepts; on $w\notin L$ it may reject **or run forever**.
* **recursive** (decidable): $L = L(M)$ for some TM $M$ that **halts on every input**.

Recursive $\subsetneq$ RE, and the witness is $A_{TM}$ (Chapter 14).

The key theorem, and the one you will actually use:

> $L$ is recursive $\iff$ both $L$ and $\overline{L}$ are RE.

The proof is a **dovetailing** argument: run both semi-deciders in parallel, one step
each in turn, and take whichever answers. One of them must.

## 2. Definitions

### Semi-decider versus decider

In [ ]:
Decider = md2mc('''TM
!! decides "the tape starts with 1" -- halts on EVERY input
I : 1 ; 1 , R -> F
I : 0 ; 0 , R -> D
I : . ; . , R -> D
''')

SemiDecider = md2mc('''TM
!! recognises "the tape starts with 1" -- LOOPS on the rest
I : 1 ; 1 , R -> F
I : 0 ; 0 , R -> L
I : . ; . , R -> L
L : 0 ; 0 , R -> L      !! the loop: never stuck, so never halts
L : 1 ; 1 , R -> L
L : . ; . , R -> L
''')

# --- thin wrappers over Jove's TM runner --------------------------------
# run_tm(T, tape, fuel) returns (truncated-paths, haltList).  A TM HALTS
# when no transition applies, and ACCEPTS if it halts in a final state.
# So an accepting state must have NO outgoing transitions, or the machine
# will run on past it.
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

def tm_tape(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return [cfg[2].rstrip('.') for cfg, _ in halts]

### Dovetailing two semi-deciders

In [ ]:
def dovetail(TA, TB, tape, maxfuel=400, step=5):
    # run both a little at a time until one of them halts
    for fuel in range(step, maxfuel + 1, step):
        if tm_halts(TA, tape, fuel=fuel): return ('A', fuel)
        if tm_halts(TB, tape, fuel=fuel): return ('B', fuel)
    return (None, maxfuel)

## 3. Tests

A **decider** halts on every input &mdash; yes or no, always.

In [ ]:
for t in ['1', '0', '11', '00']:
    print("  %-5r halts %-6s accepts %s"
          % (t, tm_halts(Decider, t), tm_accepts(Decider, t)))
assert all(tm_halts(Decider, t) for t in ['1', '0', '11', '00'])

A **semi-decider** halts on the members and loops on the rest.

In [ ]:
for t in ['1', '11', '0', '00']:
    print("  %-5r halts %-6s accepts %s"
          % (t, tm_halts(SemiDecider, t, fuel=300),
             tm_accepts(SemiDecider, t, fuel=300)))
assert tm_halts(SemiDecider, '1', fuel=50)
assert not tm_halts(SemiDecider, '0', fuel=300)
print("\nOn '0' you wait forever, and you are never told why.")

They recognise the **same** language &mdash; the difference is the guarantee.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(1, 5) for p in product('01', repeat=k)]
same = all(tm_accepts(Decider, t, fuel=200) == tm_accepts(SemiDecider, t, fuel=200)
           for t in strs)
print("same language on all %d strings? %s" % (len(strs), same))
assert same
print("\nSo this language is RECURSIVE: a halting machine for it exists.")

**Dovetailing:** run semi-deciders for $L$ and $\overline{L}$ in parallel.

In [ ]:
CoSemi = md2mc('''TM
!! recognises "the tape does NOT start with 1" -- loops on the rest
I : 0 ; 0 , R -> F
I : . ; . , R -> F
I : 1 ; 1 , R -> L
L : 0 ; 0 , R -> L
L : 1 ; 1 , R -> L
L : . ; . , R -> L
''')
for t in ['1', '0', '11', '00']:
    who, fuel = dovetail(SemiDecider, CoSemi, t)
    print("  %-5r answered by %-4s after fuel %d" % (t, who, fuel))
    assert who is not None
print("\nExactly one of the two always halts, so together they DECIDE.")

The theorem, and why the hierarchy is strict.

In [ ]:
print("L recursive  <=>  L is RE and complement(L) is RE")
print()
print("A_TM is RE (simulate the machine) but its complement is not,")
print("so A_TM is RE and NOT recursive -- which makes the containment strict.")
print("Chapter 14 does this properly.")

## 4. Exercises


1. Why must the dovetailing run both machines *interleaved* rather than one then the other?
2. Give an RE language whose complement is also RE. What does that make it?
3. Is the set of **all** TM descriptions recursive?

In [ ]:
# Your work for the exercises above.